# Wav2Vec2 自监督语音表示学习教程

本教程介绍 Wav2Vec2 自监督语音表示学习模型，包括：

1. **自监督学习原理** - 对比学习与掩码预测
2. **模型架构** - 特征编码器、Transformer、量化器
3. **预训练任务** - 对比损失与多样性损失
4. **下游任务** - CTC 语音识别、序列分类
5. **实践应用** - 模型创建与推理

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

# 添加 src 目录到路径
sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

# 检查设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 自监督学习原理

### 1.1 什么是自监督学习？

自监督学习是一种无需人工标注的学习方式，模型从数据本身构造监督信号。

**传统监督学习**：需要大量标注数据（如语音-文本对）

**自监督学习**：从未标注音频中学习通用表示

### 1.2 Wav2Vec2 的核心思想

```
原始音频 ──→ [特征编码器] ──→ 连续特征 ──→ [掩码] ──→ [Transformer]
                                  ↓                        ↓
                            [量化器]                  上下文表示
                                  ↓                        ↓
                            离散目标 ←───── 对比学习 ─────→
```

**关键步骤**：
1. 特征编码器将原始波形转换为连续特征
2. 随机掩码部分时间步
3. Transformer 从未掩码部分预测掩码位置的表示
4. 量化器将连续特征离散化作为预测目标
5. 对比损失让模型区分正确目标和干扰项

In [ ]:
# 可视化自监督学习过程
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. 原始特征
np.random.seed(42)
seq_len = 50
features = np.random.randn(seq_len)
axes[0].plot(features, 'b-', linewidth=2)
axes[0].set_title('1. 原始特征序列', fontsize=12)
axes[0].set_xlabel('时间步')

# 2. 掩码后的特征
mask_indices = [10, 11, 12, 25, 26, 27, 40, 41, 42]
masked_features = features.copy()
for idx in mask_indices:
    masked_features[idx] = 0
axes[1].plot(masked_features, 'b-', linewidth=2)
axes[1].scatter(mask_indices, [0]*len(mask_indices), c='red', s=100, zorder=5, label='掩码位置')
axes[1].set_title('2. 掩码后的特征', fontsize=12)
axes[1].set_xlabel('时间步')
axes[1].legend()

# 3. 对比学习
axes[2].bar([0, 1, 2, 3, 4], [0.9, 0.1, 0.05, 0.02, 0.01], color=['green', 'gray', 'gray', 'gray', 'gray'])
axes[2].set_xticks([0, 1, 2, 3, 4])
axes[2].set_xticklabels(['正样本', '负样本1', '负样本2', '负样本3', '负样本4'])
axes[2].set_title('3. 对比学习目标', fontsize=12)
axes[2].set_ylabel('相似度')

plt.tight_layout()
plt.show()

## 2. Wav2Vec2 模型架构

### 2.1 整体架构

```
┌─────────────────────────────────────────────────────────────────┐
│                         Wav2Vec2                                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  原始波形 [B, T]                                                 │
│      ↓                                                           │
│  ┌─────────────────┐                                            │
│  │  特征编码器      │  7层1D卷积，将16kHz音频下采样320倍          │
│  │  (CNN)          │  输出: [B, 512, T/320]                      │
│  └─────────────────┘                                            │
│      ↓                                                           │
│  ┌─────────────────┐                                            │
│  │  特征投影        │  线性层 + 层归一化                          │
│  └─────────────────┘                                            │
│      ↓                                                           │
│  ┌─────────────────┐     ┌─────────────────┐                    │
│  │  掩码 + 位置编码 │ ──→ │  Transformer    │                    │
│  └─────────────────┘     │  编码器         │                    │
│                          └─────────────────┘                    │
│      ↓                          ↓                               │
│  ┌─────────────────┐     上下文表示                              │
│  │  Gumbel 量化器   │                                            │
│  └─────────────────┘                                            │
│      ↓                                                           │
│  离散目标 (用于对比学习)                                          │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
from wav2vec2 import Wav2Vec2Config, Wav2Vec2Model, create_wav2vec2_model

# 查看默认配置
config = Wav2Vec2Config()
print("Wav2Vec2 默认配置:")
print(f"  hidden_size: {config.hidden_size}")
print(f"  num_attention_heads: {config.num_attention_heads}")
print(f"  num_hidden_layers: {config.num_hidden_layers}")
print(f"  conv_channels: {config.conv_channels}")
print(f"  num_codevectors: {config.num_codevectors}")
print(f"  num_codevector_groups: {config.num_codevector_groups}")

### 2.2 特征编码器 (Feature Encoder)

特征编码器由多层1D卷积组成，将原始波形转换为特征序列。

**设计特点**：
- 第一层使用大卷积核 (10) 捕获长程依赖
- 后续层逐渐减小卷积核
- 使用 Group Normalization 和 GELU 激活
- 总下采样率约 320 倍

In [ ]:
from wav2vec2 import FeatureEncoder

# 创建特征编码器
feature_encoder = FeatureEncoder(config)

# 模拟输入: 1秒的16kHz音频
batch_size = 2
audio_length = 16000  # 1秒
audio = torch.randn(batch_size, audio_length)

# 编码
features = feature_encoder(audio)
print(f"输入音频形状: {audio.shape}")
print(f"输出特征形状: {features.shape}")
print(f"下采样率: {audio_length / features.shape[2]:.1f}x")

In [ ]:
# 可视化卷积层结构
conv_layers = [
    ("Conv1", 10, 5, 512),
    ("Conv2", 3, 2, 512),
    ("Conv3", 3, 2, 512),
    ("Conv4", 3, 2, 512),
    ("Conv5", 3, 2, 512),
    ("Conv6", 2, 2, 512),
    ("Conv7", 2, 2, 512),
]

print("特征编码器卷积层结构:")
print("-" * 50)
print(f"{'层名':<10} {'卷积核':<10} {'步长':<10} {'输出通道':<10}")
print("-" * 50)
for name, kernel, stride, channels in conv_layers:
    print(f"{name:<10} {kernel:<10} {stride:<10} {channels:<10}")

# 计算总下采样率
total_stride = 5 * 2 * 2 * 2 * 2 * 2 * 2
print(f"\n总下采样率: {total_stride}x")

### 2.3 Transformer 编码器

Transformer 编码器处理特征序列，学习上下文表示。

**关键组件**：
- 卷积位置编码 (相对位置信息)
- 多头自注意力
- 前馈网络
- 层归一化 (Pre-LN)

In [ ]:
from wav2vec2 import TransformerEncoder

# 创建 Transformer 编码器
transformer = TransformerEncoder(config)

# 模拟输入特征
seq_len = 50
hidden_states = torch.randn(batch_size, seq_len, config.hidden_size)

# 编码
output = transformer(hidden_states)
print(f"输入形状: {hidden_states.shape}")
print(f"输出形状: {output.shape}")

# 统计参数量
num_params = sum(p.numel() for p in transformer.parameters())
print(f"Transformer 参数量: {num_params / 1e6:.2f}M")

### 2.4 Gumbel 向量量化器

量化器将连续特征离散化，作为对比学习的目标。

**工作原理**：
1. 维护多个码本 (codebook)，每个码本包含多个码向量
2. 使用 Gumbel-Softmax 进行可微分的离散选择
3. 从每个码本选择一个码向量，拼接得到最终量化表示

In [ ]:
from wav2vec2 import GumbelVectorQuantizer

# 创建量化器
quantizer = GumbelVectorQuantizer(config)

# 模拟输入
features = torch.randn(batch_size, seq_len, config.conv_channels[-1])

# 量化
quantized, perplexity = quantizer(features)
print(f"输入特征形状: {features.shape}")
print(f"量化后形状: {quantized.shape}")
print(f"困惑度 (Perplexity): {perplexity.item():.2f}")
print(f"\n码本配置:")
print(f"  码本组数: {config.num_codevector_groups}")
print(f"  每组码向量数: {config.num_codevectors}")
print(f"  总码向量组合数: {config.num_codevectors ** config.num_codevector_groups}")

## 3. 预训练任务

### 3.1 对比损失 (Contrastive Loss)

对比损失让模型学会区分正确的量化目标和干扰项。

$$\mathcal{L}_c = -\log \frac{\exp(sim(c_t, q_t)/\kappa)}{\sum_{\tilde{q} \in Q_t} \exp(sim(c_t, \tilde{q})/\kappa)}$$

其中：
- $c_t$: 上下文表示
- $q_t$: 正确的量化目标
- $Q_t$: 包含正样本和负样本的集合
- $\kappa$: 温度参数

In [ ]:
def contrastive_loss(context, quantized, negatives, temperature=0.1):
    """计算对比损失"""
    # 正样本相似度
    pos_sim = F.cosine_similarity(context, quantized, dim=-1) / temperature
    
    # 负样本相似度
    neg_sim = F.cosine_similarity(
        context.unsqueeze(2), negatives, dim=-1
    ) / temperature
    
    # 合并正负样本
    logits = torch.cat([pos_sim.unsqueeze(-1), neg_sim], dim=-1)
    
    # 目标: 第一个位置是正样本
    targets = torch.zeros(logits.shape[:-1], dtype=torch.long, device=logits.device)
    
    # 交叉熵损失
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
    return loss

# 示例
context = torch.randn(2, 50, 256)
quantized = torch.randn(2, 50, 256)
negatives = torch.randn(2, 50, 10, 256)  # 10个负样本

loss = contrastive_loss(context, quantized, negatives)
print(f"对比损失: {loss.item():.4f}")

### 3.2 多样性损失 (Diversity Loss)

多样性损失鼓励模型均匀使用所有码向量，避免码本坍塌。

$$\mathcal{L}_d = \frac{1}{GV} \sum_{g=1}^{G} -H(\bar{p}_g) = \frac{1}{GV} \sum_{g=1}^{G} \sum_{v=1}^{V} \bar{p}_{g,v} \log \bar{p}_{g,v}$$

In [ ]:
def diversity_loss(perplexity, num_codevectors):
    """计算多样性损失"""
    # 理想情况下，困惑度应该等于码向量数量
    # 这意味着所有码向量被均匀使用
    return (num_codevectors - perplexity) / num_codevectors

# 示例
perplexity = torch.tensor(200.0)  # 当前困惑度
num_codevectors = 320  # 码向量总数

d_loss = diversity_loss(perplexity, num_codevectors)
print(f"多样性损失: {d_loss.item():.4f}")
print(f"码向量利用率: {perplexity.item() / num_codevectors * 100:.1f}%")

## 4. 下游任务

### 4.1 CTC 语音识别

预训练后，可以在 Wav2Vec2 上添加 CTC 头进行语音识别微调。

In [ ]:
from wav2vec2 import Wav2Vec2ForCTC, create_wav2vec2_for_ctc

# 创建 CTC 模型
vocab_size = 32  # 字符表大小
ctc_model = create_wav2vec2_for_ctc(size="tiny", vocab_size=vocab_size)

# 模拟输入
audio = torch.randn(2, 16000)  # 1秒音频
audio_lengths = torch.tensor([16000, 14000])

# 前向传播
logits = ctc_model(audio, audio_lengths)
print(f"输入音频形状: {audio.shape}")
print(f"输出 logits 形状: {logits.shape}")
print(f"每帧预测 {vocab_size} 个字符的概率")

In [ ]:
# CTC 解码示例
def greedy_ctc_decode(logits, blank_id=0):
    """贪婪 CTC 解码"""
    # 取每帧最大概率的字符
    predictions = logits.argmax(dim=-1)
    
    # 去除重复和空白
    decoded = []
    for pred in predictions:
        chars = []
        prev = -1
        for p in pred:
            if p != prev and p != blank_id:
                chars.append(p.item())
            prev = p
        decoded.append(chars)
    return decoded

# 解码
decoded = greedy_ctc_decode(logits)
print(f"解码结果 (字符ID): {decoded}")

### 4.2 序列分类

Wav2Vec2 也可用于音频分类任务，如情感识别、说话人识别等。

In [ ]:
from wav2vec2 import Wav2Vec2ForSequenceClassification

# 创建分类模型
num_classes = 4  # 例如: 4种情感
config = Wav2Vec2Config(hidden_size=256, num_hidden_layers=4)
classifier = Wav2Vec2ForSequenceClassification(config, num_classes)

# 前向传播
logits = classifier(audio, audio_lengths)
print(f"分类 logits 形状: {logits.shape}")
print(f"预测类别: {logits.argmax(dim=-1).tolist()}")

## 5. 实践应用

### 5.1 创建不同大小的模型

In [ ]:
# 创建不同大小的模型
for size in ["tiny", "base", "large"]:
    model = create_wav2vec2_model(size)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"{size:>6} 模型参数量: {num_params / 1e6:.2f}M")

### 5.2 完整推理流程

In [ ]:
# 完整的 Wav2Vec2 推理流程
model = create_wav2vec2_model("tiny")
model.eval()

# 模拟音频输入
audio = torch.randn(1, 32000)  # 2秒音频
audio_lengths = torch.tensor([32000])

with torch.no_grad():
    # 提取特征
    features = model(audio, audio_lengths)
    
print(f"输入: {audio.shape}")
print(f"输出特征: {features.shape}")
print(f"特征维度: {features.shape[-1]}")

## 总结

### Wav2Vec2 的关键创新

1. **自监督预训练**: 从大量未标注音频学习通用表示
2. **对比学习**: 通过区分正负样本学习有意义的特征
3. **向量量化**: 将连续特征离散化，提供稳定的学习目标
4. **高效微调**: 预训练模型只需少量标注数据即可达到优秀性能

### 应用场景

- 语音识别 (ASR)
- 说话人识别
- 情感识别
- 语音翻译
- 音频事件检测